# ML Honors Assignment - 4

**Name:** Vaishnav Mankar

**Rollno:** 41252

---

**Problem Statement:** Text classification for Sentimental analysis using KNN. Note: Use twitter data.

**Dataset:** Twitter US Airline Sentiment (Kaggle — `Tweets.csv`)

**Requirements:** Python 3, Pandas, Scikit-Learn

**Code:-**

In [1]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("data/Tweets.csv")[["text", "airline_sentiment"]]
print("Dataset Shape:", df.shape)
print(df["airline_sentiment"].value_counts())
display(df.head())

Dataset Shape: (14640, 2)
airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64


,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [2]:
def clean(text):
    text = re.sub(r"http\S+|@\w+", " ", text.lower())
    return re.sub(r"[^a-z\s]", " ", text)


df["clean"] = df["text"].apply(clean)
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["airline_sentiment"], test_size=0.2, stratify=df["airline_sentiment"], random_state=42)

tfidf = TfidfVectorizer(stop_words="english", min_df=2, max_features=5000)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)
print("TF-IDF matrix:", X_train_vec.shape)

TF-IDF matrix: (11712, 4766)


In [3]:
scores = {}
for k in (1, 3, 5, 7, 11, 15):
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    scores[k] = cross_val_score(knn, X_train_vec, y_train, cv=3).mean()
    print(f"K = {k:>2}: cross-validation accuracy {scores[k]:.3f}")
best_k = max(scores, key=scores.get)
print("Best K:", best_k)

K =  1: cross-validation accuracy 0.593
K =  3: cross-validation accuracy 0.667
K =  5: cross-validation accuracy 0.687
K =  7: cross-validation accuracy 0.699
K = 11: cross-validation accuracy 0.712
K = 15: cross-validation accuracy 0.723
Best K: 15


In [4]:
knn = KNeighborsClassifier(n_neighbors=best_k, metric="cosine").fit(X_train_vec, y_train)
pred = knn.predict(X_test_vec)
print(f"Test Accuracy: {accuracy_score(y_test, pred) * 100:.2f}%\n")
print(classification_report(y_test, pred))

Test Accuracy: 72.10%

              precision    recall  f1-score   support

    negative       0.77      0.89      0.82      1835
     neutral       0.52      0.37      0.43       620
    positive       0.69      0.53      0.60       473

    accuracy                           0.72      2928
   macro avg       0.66      0.60      0.62      2928
weighted avg       0.70      0.72      0.70      2928



In [5]:
samples = [
    "Great flight, the crew was lovely and friendly",
    "Worst airline ever, my bag was lost and the delay was awful",
    "What time does the flight to Delhi board",
]
for tweet, label in zip(samples, knn.predict(tfidf.transform([clean(t) for t in samples]))):
    print(f"{label:>9}  <-  {tweet}")

 positive  <-  Great flight, the crew was lovely and friendly
 negative  <-  Worst airline ever, my bag was lost and the delay was awful
 negative  <-  What time does the flight to Delhi board
